In [5]:
#!pip install timecopilot


In [7]:
import pandas as pd

from timecopilot import TimeCopilotForecaster
from timecopilot.models.foundation.chronos import Chronos
from timecopilot.models.foundation.moirai import Moirai
from timecopilot.models.foundation.timesfm import TimesFM

from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mape, rmse, mae, bias

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
2026-05-09 18:21:38.639550: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-09 18:21:38.639588: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-09 18:21:38.640770: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-09 18:21:39.209735: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [9]:
df = (
    pd.read_parquet('/work/sample_hotels-1.parquet')
    [['unique_id', 'ds', 'y']]
)

df['ds'] = pd.to_datetime(df['ds'])

# Drop unreliable hotels (same as other notebooks)
df = df.query("unique_id not in ['hotel_77', 'hotel_28']")

In [11]:
tcf = TimeCopilotForecaster(
    models = [
        Chronos(repo_id="amazon/chronos-bolt-small"),
        Moirai(),
        TimesFM(
            repo_id="google/timesfm-2.5-200m-pytorch",
            alias="TimesFM-2.5"
        ),
    ]
)

In [13]:
train_df = df.groupby('unique_id').apply(lambda x: x.iloc[:-28]).reset_index(drop=True)


In [ ]:
t_cv = tcf.cross_validation(
    df=train_df,
    h = 28,
    freq = 'D',
    n_windows = 5,
    step_size = 28
)

0it [00:00, ?it/s]`torch_dtype` is deprecated! Use `dtype` instead!
`torch_dtype` is deprecated! Use `dtype` instead!

100%|██████████| 2/2 [00:00<00:00,  6.66it/s]
1it [00:03,  3.96s/it]
100%|██████████| 2/2 [00:00<00:00,  6.49it/s]
2it [00:04,  2.13s/it]
100%|██████████| 2/2 [00:00<00:00,  6.25it/s]
3it [00:05,  1.55s/it]
100%|██████████| 2/2 [00:00<00:00,  6.27it/s]
4it [00:06,  1.31s/it]
100%|██████████| 2/2 [00:00<00:00,  5.75it/s]
5it [00:07,  1.50s/it]
0it [00:00, ?it/s]
0it [00:00, ?it/s]

In [21]:
eval_transformers = evaluate(
    df = t_cv,
    metrics = [mape, rmse, mae, bias],
    models = ['Chronos', 'Moirai', 'TimesFM-2.5']
)

eval_transformers = (
    eval_transformers
    .drop(columns=['cutoff'])
    .groupby(['unique_id', 'metric'])
    .mean()
    .reset_index()
)

display(eval_transformers)

,unique_id,metric,Chronos,Moirai,TimesFM-2.5
0,hotel_0,bias,0.044525,0.097055,0.055096
1,hotel_0,mae,0.165588,0.181458,0.146745
2,hotel_0,mape,0.282946,0.322964,0.254261
3,hotel_0,rmse,0.200802,0.221511,0.183185
4,hotel_105,bias,-0.000151,0.012878,-0.003361
...,...,...,...,...,...
63,hotel_91,rmse,0.187032,0.192697,0.192446
64,hotel_98,bias,-0.037134,-0.048425,-0.048646
65,hotel_98,mae,0.104254,0.101779,0.100166
66,hotel_98,mape,0.431316,0.399043,0.386054


In [23]:
models = ['Chronos', 'Moirai', 'TimesFM-2.5']


In [25]:
models = ['Chronos', 'Moirai', 'TimesFM-2.5']

# ensure numeric
eval_transformers[models] = eval_transformers[models].apply(pd.to_numeric, errors='coerce')

# pick winner (lowest error per row)
eval_transformers['winner'] = eval_transformers[models].idxmin(axis=1)

In [27]:

win_counts = (
    eval_transformers
    .groupby(['metric', 'winner'])
    .size()
    .reset_index(name='wins')
)

display(win_counts)

,metric,winner,wins
0,bias,Chronos,7
1,bias,Moirai,5
2,bias,TimesFM-2.5,5
3,mae,Chronos,6
4,mae,TimesFM-2.5,11
5,mape,Chronos,5
6,mape,Moirai,1
7,mape,TimesFM-2.5,11
8,rmse,Chronos,7
9,rmse,Moirai,1


### Final testing outputs

In [29]:
# Reload the full dataset
df = (
    pd.read_parquet('/work/sample_hotels-1.parquet')
    [['unique_id', 'ds', 'y']]
)
df['ds'] = pd.to_datetime(df['ds'])
df = df.query("unique_id not in ['hotel_77', 'hotel_28']")

# Now recreate train/test split
train_df = df.groupby('unique_id').apply(lambda x: x.iloc[:-28]).reset_index(drop=True)
test_df = df.groupby('unique_id').tail(28).reset_index(drop=True)

In [35]:
test_df = df.groupby('unique_id').tail(28).reset_index(drop=True)

# TimeCopilot predicts directly without a separate fit
forecast_df = tcf.forecast(
    df=train_df,
    h=28,
    freq='D'
)

# Merge with actuals
results_test = forecast_df.merge(test_df, on=['unique_id', 'ds'])

# Evaluate on test set
eval_test = evaluate(
    df=results_test,
    metrics=[mape, rmse, mae, bias],
    models=['Chronos', 'Moirai', 'TimesFM-2.5']
)

# Save to CSV
eval_test.to_csv('timescopilot_test_eval.csv', index=False)
results_test.to_csv('timescopilot_test_forecasts.csv', index=False)

display(eval_test)

100%|██████████| 2/2 [00:00<00:00,  4.01it/s]
17it [00:24,  1.47s/it]
100%|██████████| 1/1 [00:14<00:00, 14.95s/it]


,unique_id,metric,Chronos,Moirai,TimesFM-2.5
0,hotel_0,mape,0.132146,0.156362,0.100845
1,hotel_105,mape,0.057337,0.046491,0.047040
2,hotel_112,mape,0.109835,0.108670,0.101394
3,hotel_126,mape,0.017834,0.013403,0.012292
4,hotel_133,mape,0.052609,0.050388,0.048501
...,...,...,...,...,...
63,hotel_7,bias,0.110739,0.122996,0.114468
64,hotel_70,bias,-0.141890,-0.187607,-0.105320
65,hotel_84,bias,0.043260,0.013216,0.005225
66,hotel_91,bias,0.005335,0.041123,-0.023100


In [1]:
import matplotlib.pyplot as plt
import os

for hotel_id in results_test['unique_id'].unique():
    actual  = test_df[test_df['unique_id'] == hotel_id]
    preds   = results_test[results_test['unique_id'] == hotel_id]
    history = train_df[train_df['unique_id'] == hotel_id].tail(30)

    plt.figure(figsize=(16, 6))
    plt.plot(history['ds'], history['y'], label='Historical')
    plt.plot(actual['ds'], actual['y'], label='Actual')
    plt.plot(preds['ds'], preds['Chronos'], label='Chronos')
    plt.plot(preds['ds'], preds['Moirai'], label='Moirai')
    plt.plot(preds['ds'], preds['TimesFM-2.5'], label='TimesFM-2.5')
    plt.title(hotel_id)
    plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()
  

NameError: name 'results_test' is not defined

Three zero\-shot foundation models were evaluated using 5\-fold time series cross\-validation and a final 28\-day held\-out test set across 17 hotel series\.
TimesFM\-2\.5 was the strongest overall model, winning the most cross\-validation windows on both MAE \(11 wins\) and MAPE \(11 wins\), and this was confirmed on the final test set where it again achieved the lowest error on most series\.
Chronos showed the least bias with 7 ME wins, meaning it was most centered around the actual values on average, though it fell behind TimesFM\-2\.5 on raw accuracy\.
Moirai was the weakest performer, winning only 1 window each on MAPE and RMSE, suggesting it is less well\-suited for hotel occupancy forecasting\.


Performance varied noticeably across series\. Hotels like hotel\_126, hotel\_133, and hotel\_21 were easy to forecast with very low MAPE across all models, likely reflecting stable and predictable occupancy patterns\. In contrast, hotel\_7, hotel\_35, hotel\_42, and hotel\_98 had consistently high MAPE values above 0\.29 regardless of model, suggesting these hotels have more volatile or irregular occupancy that is harder to capture

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=94252149-99d5-4a3a-8951-54d7f526c43a' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>